# Sales Prediction Using Python and Machine Learning

This notebook demonstrates how to predict product sales based on advertising expenditure across different platforms (TV, Radio, Newspaper).

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("Advertising.csv")
df.head()

In [ ]:
df.tail()

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 3: Data Cleaning
Check for missing values, duplicates, and remove unnecessary columns (like `Unnamed: 0`).

In [ ]:
print("Missing Values:\n", df.isnull().sum())

In [ ]:
print("Duplicate Records:", df.duplicated().sum())

In [ ]:
# Remove unnecessary index column if it exists
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.head()

## Step 4: Exploratory Data Analysis
### A. Distribution Analysis

In [ ]:
plt.figure(figsize=(15, 10))
plt.suptitle('Advertising Spending Distribution', fontsize=16)

plt.subplot(2, 2, 1)
sns.histplot(df['TV'], kde=True, color='blue')
plt.title('TV Advertising Distribution')
plt.xlabel('TV')
plt.ylabel('Frequency')

plt.subplot(2, 2, 2)
sns.histplot(df['Radio'], kde=True, color='green')
plt.title('Radio Advertising Distribution')
plt.xlabel('Radio')
plt.ylabel('Frequency')

plt.subplot(2, 2, 3)
sns.histplot(df['Newspaper'], kde=True, color='orange')
plt.title('Newspaper Advertising Distribution')
plt.xlabel('Newspaper')
plt.ylabel('Frequency')

plt.subplot(2, 2, 4)
sns.histplot(df['Sales'], kde=True, color='red')
plt.title('Sales Distribution')
plt.xlabel('Sales')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

### B. Scatter Plots
Understanding the relationship between advertising expenditure and sales.

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(x='TV', y='Sales', data=df, color='blue')
plt.title('TV Advertising vs Sales')

plt.subplot(1, 3, 2)
sns.scatterplot(x='Radio', y='Sales', data=df, color='green')
plt.title('Radio Advertising vs Sales')

plt.subplot(1, 3, 3)
sns.scatterplot(x='Newspaper', y='Sales', data=df, color='orange')
plt.title('Newspaper Advertising vs Sales')

plt.tight_layout()
plt.show()

### C. Correlation Analysis

In [ ]:
correlation = df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

**Observation**: TV advertising has the strongest positive correlation with Sales (0.78), indicating that higher spending on TV ads is closely associated with higher sales.

## Step 5: Feature Selection & Train-Test Split

In [ ]:
# Features (X) and Target (y)
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

# Splitting the dataset into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

## Step 6: Machine Learning Model
Training a Multiple Linear Regression model.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## Step 7: Model Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Evaluation Metrics for Linear Regression:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

## Step 8: Actual vs Predicted Sales

In [ ]:
comparison = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred
})
print(comparison.head(10))

plt.figure(figsize=(10, 6))
plt.plot(range(len(y_test)), y_test.values, label='Actual Sales', marker='o')
plt.plot(range(len(y_test)), y_pred, label='Predicted Sales', marker='x')
plt.title('Actual vs Predicted Sales')
plt.xlabel('Data Points')
plt.ylabel('Sales')
plt.legend()
plt.show()

## Step 9: Advertising Impact Analysis
Understanding the effect of each advertising channel.

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})
print(coefficients)

plt.figure(figsize=(8, 5))
sns.barplot(x="Feature", y="Coefficient", data=coefficients, palette="viridis")
plt.title("Advertising Channel Coefficient Comparison")
plt.ylabel("Coefficient Value")
plt.xlabel("Advertising Channel")
plt.show()

**Explanation**: The regression coefficients represent the model's estimated association between each platform and sales, holding the other variables constant. They indicate that spending on Radio has a strong positive influence on predicted sales per unit of budget. Important: Correlation and regression do not necessarily prove direct causation.

## Step 10: Compare Multiple Models
Let's evaluate Linear Regression, Decision Tree, and Random Forest.

In [ ]:
# Initialize models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42)
}

results = []

for name, m in models.items():
    m.fit(X_train, y_train)
    predictions = m.predict(X_test)
    
    m_mae = mean_absolute_error(y_test, predictions)
    m_rmse = np.sqrt(mean_squared_error(y_test, predictions))
    m_r2 = r2_score(y_test, predictions)
    
    results.append([name, m_mae, m_rmse, m_r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R² Score"])
print(results_df.to_markdown(index=False))

**Conclusion**: Random Forest typically provides the best R² Score and the lowest Errors on this dataset due to its ability to capture non-linear interactions between advertising channels.

## Step 11: Sales Prediction System

In [ ]:
# Example of User Input Prediction (Uncomment to use interactively)
"""
tv = float(input("Enter TV advertising budget: "))
radio = float(input("Enter Radio advertising budget: "))
newspaper = float(input("Enter Newspaper advertising budget: "))

new_data = pd.DataFrame({
    "TV": [tv],
    "Radio": [radio],
    "Newspaper": [newspaper]
})

prediction = model.predict(new_data)
print(f"Predicted Sales: {prediction[0]:.2f}")
"""
print("Run Streamlit app (app.py) for the interactive UI system.")